1. “Suggest a Christopher Nolan Movie”
intent: recommend
type: movie
director: Christopher Nolan
2. “I want to watch a comedy movie while having my lunch”
intent: recommend
type: movie
genre: comedy
extra nuance: “while having my lunch” → maybe treat as “something light / not too long”
could map to: duration < some threshold, or just ignore at first.
3. “Suggest a good horror movie”
intent: recommend
type: movie (implicit)
genre: horror
“good” → maybe prefer higher‑rated / popular ones, or just pick a strong candidate.
4. “Suggest a TV show to watch. I am in the mood for some thriller”
intent: recommend
type: TV Show
genre: thriller
5. “Tell about a famous Indian Movie that I can watch”
intent: recommend + describe
type: movie
country: India
“famous” → prefer popular/well‑known (you might approximate via recency or presence in description, but at the start you can just pick a random one).
Response should include a short description, not just the title.
6. “Suggest a Robert Downey Jr movie”
intent: recommend
type: movie
actor in cast: Robert Downey Jr
7. “I want to see some classic American movie”
intent: recommend
type: movie
country: United States (or contains “United States” / “USA”)
“classic” → translate to an older release year range, e.g. before 2000, or before 1990.
8. “Suggest a comedy tv show”
intent: recommend
type: TV Show
genre: comedy


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn

In [2]:
df = pd.read_csv("/Users/jaimeet/Documents/Movie Recommender/data/netflix_titles.csv")
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [4]:
for c in df.columns:
    if df[c].isnull().any():
        df[c] = df[c].fillna('unknown')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      8807 non-null   object
 4   cast          8807 non-null   object
 5   country       8807 non-null   object
 6   date_added    8807 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8807 non-null   object
 9   duration      8807 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [6]:
df.shape

(8807, 12)

Columns I need: title,director,cast,country,release_year,rating,duration,description.

In [7]:
df['overview'] = df['title'] + " " + df['listed_in'] + " " + df['description'] + " " + df['cast']

In [8]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",Dick Johnson Is Dead Documentaries As her fath...
1,s2,TV Show,Blood & Water,unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...","Blood & Water International TV Shows, TV Drama..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,"Ganglands Crime TV Shows, International TV Sho..."
3,s4,TV Show,Jailbirds New Orleans,unknown,unknown,unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...","Jailbirds New Orleans Docuseries, Reality TV F..."
4,s5,TV Show,Kota Factory,unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,"Kota Factory International TV Shows, Romantic ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...","Zodiac Cult Movies, Dramas, Thrillers A politi..."
8803,s8804,TV Show,Zombie Dumb,unknown,unknown,unknown,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...","Zombie Dumb Kids' TV, Korean TV Shows, TV Come..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,"Zombieland Comedies, Horror Movies Looking to ..."
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...","Zoom Children & Family Movies, Comedies Dragge..."


In [9]:
df[(df['type'] == 'Movie') & (df['director'].str.lower().str.contains("nolan"))]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
340,s341,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...","United States, United Kingdom","August 1, 2021",2010,PG-13,148 min,"Action & Adventure, Sci-Fi & Fantasy, Thrillers",A troubled thief who extracts secrets from peo...,"Inception Action & Adventure, Sci-Fi & Fantasy..."


In [10]:
df[(df['listed_in'].str.contains('Action & Adventure, Sci-Fi & Fantasy, Thrillers')) & (df['type'] == 'Movie')]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
340,s341,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...","United States, United Kingdom","August 1, 2021",2010,PG-13,148 min,"Action & Adventure, Sci-Fi & Fantasy, Thrillers",A troubled thief who extracts secrets from peo...,"Inception Action & Adventure, Sci-Fi & Fantasy..."


In [22]:
df[df['title'].str.contains("To All the Boys")]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
732,s733,Movie,To All the Boys: Always and Forever - The Afte...,unknown,"Cast members of the ""To All the Boys"" films di...",unknown,"June 12, 2021",2021,TV-MA,36 min,Movies,"Cast members of the ""To All the Boys"" films di...",To All the Boys: Always and Forever - The Afte...
2925,s2926,Movie,To All the Boys: P.S. I Still Love You,Michael Fimognari,"Lana Condor, Noah Centineo, Jordan Fisher, Ann...",United States,"February 12, 2020",2020,TV-14,102 min,"Comedies, Romantic Movies","Lara Jean is officially Peter’s girlfriend, so...",To All the Boys: P.S. I Still Love You Comedie...
4698,s4699,Movie,To All the Boys I’ve Loved Before,Susan Johnson,"Lana Condor, Noah Centineo, Janel Parrish, Ann...",United States,"August 17, 2018",2018,TV-14,100 min,"Comedies, International Movies, Romantic Movies",When her secret love letters somehow get maile...,"To All the Boys I’ve Loved Before Comedies, In..."


In [19]:
df[df['cast'].str.contains("nolan",case=False,na=False)]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
1845,s1846,Movie,Batman: The Killing Joke,Sam Liu,"Kevin Conroy, Mark Hamill, Tara Strong, Ray Wi...",United States,"October 15, 2020",2016,R,77 min,Action & Adventure,The Joker makes life hell for the Gordon famil...,Batman: The Killing Joke Action & Adventure Th...
1867,s1868,Movie,House of the Witch,Alex Merkin,"Emily Bader, Darren Mann, Michelle Randolph, C...",United States,"October 11, 2020",2017,TV-MA,86 min,Horror Movies,A group of daring teens finds themselves in a ...,House of the Witch Horror Movies A group of da...
2249,s2250,TV Show,The Epic Tales of Captain Underpants in Space,unknown,"Nat Faxon, Jay Gragnani, Ramone Hamilton, Sean...",United States,"July 10, 2020",2020,TV-Y7,1 Season,"Kids' TV, TV Action & Adventure, TV Comedies",Best friends George and Harold — along with th...,The Epic Tales of Captain Underpants in Space ...
2927,s2928,Movie,Captain Underpants Epic Choice-o-Rama,unknown,"Nat Faxon, Jay Gragnani, Ramone Hamilton, Sean...",United States,"February 11, 2020",2020,TV-Y7,81 min,"Children & Family Movies, Comedies","In this interactive special, Harold and George...",Captain Underpants Epic Choice-o-Rama Children...
3172,s3173,Movie,Spirit Riding Free: Spirit of Christmas,unknown,"Amber Frank, Bailey Gambertoglio, Sydney Park,...",United States,"December 6, 2019",2019,TV-Y7,46 min,Children & Family Movies,Lucky and friends must figure out how to get h...,Spirit Riding Free: Spirit of Christmas Childr...
3406,s3407,TV Show,Spirit Riding Free: Pony Tales,unknown,"Amber Frank, Bailey Gambertoglio, Sydney Park,...",United States,"October 18, 2019",2019,TV-Y7,2 Seasons,Kids' TV,"Find the fun and adventure of ""Spirit Riding F...",Spirit Riding Free: Pony Tales Kids' TV Find t...
3943,s3944,TV Show,Spirit: Riding Free,unknown,"Amber Frank, Sydney Park, Bailey Gambertoglio,...",United States,"April 5, 2019",2019,TV-Y7,8 Seasons,Kids' TV,"In a small Western town, spunky ex-city girl L...",Spirit: Riding Free Kids' TV In a small Wester...
4138,s4139,Movie,Lego DC Comics: Batman Be-Leaguered,Rick Morales,"Dee Bradley Baker, Troy Baker, John DiMaggio, ...",United States,"February 1, 2019",2014,TV-Y7,22 min,Movies,When Superman and the other Justice League sup...,Lego DC Comics: Batman Be-Leaguered Movies Whe...
4762,s4763,TV Show,Home: Adventures with Tip & Oh,unknown,"Rachel Crow, Mark Whitten, Ana Ortiz, Ron Func...",United States,"July 20, 2018",2018,TV-Y7,4 Seasons,"Kids' TV, TV Comedies",A misfit alien named Oh moves in with Tip and ...,"Home: Adventures with Tip & Oh Kids' TV, TV Co..."
5827,s5828,TV Show,LEGO Bionicle: The Journey to One,unknown,"Nolan Balzer, Paolo Bryant, Jacqui Fox, Quinn ...",United States,"July 29, 2016",2016,TV-Y7,2 Seasons,Kids' TV,Six legendary heroes find themselves on an epi...,LEGO Bionicle: The Journey to One Kids' TV Six...


In [13]:
df.to_csv("/Users/jaimeet/Documents/Movie Recommender/data/netflix_cleaned.csv",index=False)